# Anomaly Detection using GNN
CERT Insider Threat Dataset - cleaning and Exploratory Data Analysis (EDA) 
Step 1: Preparing Data for Graph and GNN Model Building 

Author: Dagmara Krenich

This file contains the full cleaning and analysis pipeline for all 5 CERT v4.1 dataset files. We use PySpark due to huge data size (over 14 GB).

1. Required libraries

In [6]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import TimestampType
from pyspark.sql.window import Window

2. PySpark Initialization - PySpark allows you to process data larger than available RAM by loading it in batches, without loading the entire thing into memory. "local[*]" mode = all CPU cores.

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CERT Analysis") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.executorEnv.SPARK_LOCAL_IP", "127.0.0.1") \
    .getOrCreate()


## Section A: Helpers
A set of functions that will be used for each CERT data file. 

In [ ]:
def code_quality(df, name):
    '''
    Displays a basic quality report for the data frame:
    - number of records
    - number of columns
    - number and percentage of missing values ​​in each column
    - number of duplicates
    '''
    print(f"Data quality for: {name}")

    n_rows = df.count()
    n_cols = len(df.columns)
    print(f"Number of records: {n_rows}, number of columns: {n_cols}")

    # Missing values - for each column if null or "" -> 1, else 0, return sum
    missing_exprs = [
        F.sum(
            F.when(
                F.col(c).isNull() | (F.col(c) == ""), 1
            ).otherwise(0)
        ).alias(c)
        for c in df.columns
    ]

    missing_df = df.select(missing_exprs)

    for row in missing_df.collect():
        for col_name in df.columns:
            nulls = row[col_name]
            if nulls > 0:
                pct = 100 * nulls / n_rows
                print(f"{col_name:<25} {nulls:>8,} ({pct:.2f}%)")

    # Missing Values
    n_distinct = df.dropDuplicates().count()
    n_dup = n_rows - n_distinct

    print(f"\nDuplicates: {n_dup:,} ({100*n_dup/n_rows:.2f}%)")

    return {
        "rows": n_rows,
        "columns": n_cols,
        "duplicates": n_dup,
        "missing": missing_df
    }

In [18]:
df = spark.read.csv("../data/logon.csv", header=True, inferSchema=True)
df_cache = df.cache()

In [21]:
# test
code_quality(df_cache, "logon.csv")

Data quality for: logon.csv
Number of records: 899118, number of columns: 5

Missing values: DataFrame[id: bigint, date: bigint, user: bigint, pc: bigint, activity: bigint]

Duplicates: 0 (0.00%)


{'rows': 899118,
 'columns': 5,
 'duplicates': 0,
 'missing': DataFrame[id: bigint, date: bigint, user: bigint, pc: bigint, activity: bigint]}